**This notebook shows the launch of the SVC model on the 100k comments and the application of HCS**

In [ ]:
import pandas as pd
import numpy as np
import regex as re

import nltk
from nltk.corpus import wordnet
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import textwrap
from sklearn.preprocessing import OneHotEncoder
import string
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

import swifter
from sklearn.svm import SVC

import numpy as np


nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package wordnet to /Users/emma/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/emma/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /Users/emma/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/emma/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [87]:
labeled = pd.read_csv('/Users/emma/Desktop/labeled_data_B.csv')
labeled = labeled[labeled['my_label'] != 'unclear'].reset_index(drop=True)
labeled['my_label'].value_counts()

my_label
no stance    1758
prochoice    1012
prolife       760
Name: count, dtype: int64

In [88]:
labeled['preprocessed'] = (
    labeled['body']
    .str.replace(f"[{string.punctuation}]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

In [ ]:
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(word):

    tag = pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

def lemmatize_text(text):
    if isinstance(text, str):
        words = word_tokenize(text)
        lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in words]
        return " ".join(lemmatized_words)
    return text

In [89]:
labeled['preprocessed'] = labeled['body'].swifter.apply(lemmatize_text)

Pandas Apply:   0%|          | 0/3530 [00:00<?, ?it/s]

In [72]:
labeled['id'].duplicated().sum()

0

In [90]:
# encoding labels
label_encoder = LabelEncoder()
y  = label_encoder.fit_transform(labeled['my_label'])
(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

{'no stance': 0, 'prochoice': 1, 'prolife': 2}

In [104]:
tfidf = TfidfVectorizer(max_features = 1200,
                        ngram_range = (1,2),
                        stop_words='english'
                        )

X  = tfidf.fit_transform(labeled['preprocessed'])

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=13)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=13)

svm_model = SVC(kernel="linear", 
                C= .3, 
                probability=True, 
                 class_weight={0: 1.0, 1: 3.0, 2: 2.5})

svm_model.fit(X_train, y_train)

y_train_pred = svm_model.predict(X_train)
y_val_pred = svm_model.predict(X_val)

print("Train Acc:", accuracy_score(y_train, y_train_pred))
print('Validation Acc:', accuracy_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred, target_names=label_encoder.classes_))
print("")

scores = cross_val_score(svm_model, X_train, y_train, cv=5, scoring='accuracy')
print("CV Scores:", scores)
print("CV Std Dev:", np.std(scores))

(confusion_matrix(y_val, y_val_pred))

Train Acc: 0.8535006070416835
Validation Acc: 0.8109640831758034
              precision    recall  f1-score   support

   no stance       0.93      0.80      0.86       263
   prochoice       0.68      0.83      0.75       152
     prolife       0.79      0.81      0.80       114

    accuracy                           0.81       529
   macro avg       0.80      0.81      0.80       529
weighted avg       0.83      0.81      0.81       529


CV Scores: [0.79393939 0.74898785 0.77327935 0.79352227 0.81174089]
CV Std Dev: 0.02144468578244969


array([[211,  42,  10],
       [ 11, 126,  15],
       [  5,  17,  92]])

In [ ]:
CV = [0.79393939, 0.74898785, 0.77327935, 0.79352227, 0.81174089]

In [105]:
y_test_pred = svm_model.predict(X_test)
print("test acc:", accuracy_score(y_test, y_test_pred))

test acc: 0.8


80% accuracy, model seems stable between CV runs. \n

Ran this model through a stratified sample of subreddits, hoping that this would allow for some PL comments to make it in the sample.

- 0    78411
- 1    16286
- 2     5303

Sampled no stance and it seemed to be highly accurate, which is unsuprising given how well the model was preforming. PC was struggling, PL seemed to be okay. This seemed be a repeat of the model preformance (above). really good representation of strengths and weakenesses we saw above.

however, for topic modeling and after manually reviewing the returned classifications, i dont feel comfortable with this dataset. i will be setting a confidence threshold and viewing the model preformqance, than iteratively building a dataset that way. this will not be able to show me the polarization/dist of discussions but will be a far more accurate representation of what the various sides are discussing.

In [130]:
y_proba = svm_model.predict_proba(X_test)


In [131]:
conf_thres = .8

hcp = np.max(y_proba, axis = 1) >= conf_thres
hcl = np.argmax(y_proba, axis=1)[hcp]
ytf = y_test[hcp]


In [136]:
hc_accuracy = accuracy_score(ytf, hcl)
print("High-Confidence Test Accuracy:", hc_accuracy)
print(classification_report(ytf, hcl, target_names=label_encoder.classes_))

High-Confidence Test Accuracy: 0.92
              precision    recall  f1-score   support

   no stance       0.97      0.95      0.96       184
   prochoice       0.88      0.82      0.85        65
     prolife       0.80      0.96      0.88        51

    accuracy                           0.92       300
   macro avg       0.89      0.91      0.89       300
weighted avg       0.92      0.92      0.92       300



- 92% accuracy is a much better preformannce. all the classes were helped by this. going to artifically build and balance dataset by iteratevely running samples through this.

In [249]:
unlabeled = pd.read_parquet('/Users/emma/Desktop/thesis/actual_folder/clean/total_comments_B.parquet')

unlabeled = unlabeled[~unlabeled['id'].isin(hcs['id'])]

unlabeled.shape

(2437547, 9)

In [ ]:
sample_size = 50000

sample_politics = unlabeled[unlabeled['subreddit'] == 'politics'].sample(n=sample_size, random_state=13)
sample_conservative = unlabeled[unlabeled['subreddit'] == 'Conservative'].sample(n=sample_size, random_state=13)

stratified_sample = pd.concat([sample_politics, sample_conservative])


In [280]:
stratified_sample['preprocessed'] = (
    stratified_sample['body']
    .str.replace(f"[{string.punctuation}]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)
stratified_sample['preprocessed'] = stratified_sample['preprocessed'].swifter.apply(lemmatize_text)

Pandas Apply:   0%|          | 0/100000 [00:00<?, ?it/s]

In [281]:
X_unlabeled = tfidf.transform(stratified_sample['preprocessed'])

In [ ]:
proba = svm_model.predict_proba(X_unlabeled)

threshold = .8

y_pred = np.argmax(proba, axis =1)

max_proba = proba.max(axis=1)

conf_pred = max_proba > threshold

uncertain = ~conf_pred

y_pred[uncertain] = -1

In [283]:
stratified_sample['predicted_label'] = y_pred

In [284]:
stratified_sample['predicted_label'].value_counts()

predicted_label
 0    68501
-1    29916
 1     1100
 2      483
Name: count, dtype: int64

predicted_label
- 0    80454
- -1    12515
- 1     4426
- 2     2605
Name: count, dtype: int64

majority of runs looked like this. 

In [ ]:
# highconfidence_samples = labeled.copy()

In [ ]:
# ns_hc = stratified_sample[stratified_sample ['predicted_label'] == 0].sample(25000)

In [285]:
pc_hc = stratified_sample[stratified_sample['predicted_label'] == 1]
pl_hc = stratified_sample[stratified_sample['predicted_label'] == 2]

In [286]:
# hcs = pd.concat([highconfidence_samples, ns_hc, pc_hc, pl_hc]).reset_index(drop=True)

hcs = pd.concat([hcs, pc_hc, pl_hc]).reset_index(drop=True)

In [287]:
hcs['predicted_label'].value_counts()

predicted_label
0.0    25565
1.0    16613
2.0    10838
Name: count, dtype: int64

In [294]:
hcs.to_csv('tm_hcs.csv', index = False)